# 🏛️ CausalNerve: Synthetic Engine Quickstart
This interactive notebook demonstrates the real-time causal graph mutating over a synthetic engine degradation stream. We use a 100% synthetic dataset so there are no external dependencies or downloads required.

In [ ]:
!pip install causalnerve==1.0.5 causalnerve-observe==1.0.5

In [ ]:
import time
import numpy as np
from causalnerve import CausalNerve
from causalnerve.datasets import SyntheticStreamGenerator
from causalnerve.memory import StructuralReplayEngine, GraphDiff
from causalnerve_observe import observe

# 1. Setup CausalNerve Engine
nerve = CausalNerve(nodes=6, state_dim=32)
replay = StructuralReplayEngine(snapshot_interval=5)

# 2. Load Synthetic Engine Data
print("Generating stable synthetic baseline...")
historical_data = np.array(list(SyntheticStreamGenerator.stable(n_cycles=150)))

# Train baseline DAG structure
nerve.fit(historical_data)
print("Engine learned baseline DAG structure.")

In [ ]:
# 3. Stream Real-Time Telemetry (Injecting Degradation Drift)
print("Streaming degrading engine telemetry...")
streaming_data = np.array(list(SyntheticStreamGenerator.with_drift(n_cycles=50)))

for cycle in range(50):
    res = nerve.step(streaming_data[cycle])
    
    if cycle % 5 == 0:
        # In a real environment, you extract nerve.adjacency_matrix
        # For immediate UI wow-factor, we use a mock edge set to simulate structure fracture
        adj_list = [(0, 1, 0.8), (3, 1, 0.5), (4, 2, 0.6)] if cycle < 25 else [(0, 1, 0.2), (3, 1, 0.9), (4, 2, 0.1), (5, 0, 0.7)]
        replay.record_snapshot(cycle, adj_list, res.leakage, 3.0)

print("Telemetry ingested. Mutations recorded.")

In [ ]:
# 4. Launch the WebGL Dashboard
# Fix GraphDiff v1.0.5 API drift for UI rendering
if not hasattr(GraphDiff, "edges_stable"):
    GraphDiff.edges_stable = property(lambda s: getattr(s, "stable_edges", []))
for snap in replay.snapshots:
    if not hasattr(snap, "active_alarms"):
        snap.active_alarms = []

# Force standard SVG Plotly rendering (fixes Colab WebGL blank graph bug)
import plotly.graph_objects as go
go.Scattergl = go.Scatter

nerve.replay_engine = replay
nerve.current_cycle = 50
nerve.preset_name = "Synthetic Engine 001"
nerve.node_labels = {0: "Fan_Speed", 1: "LPC_Pres", 2: "HPC_Pres", 3: "LPT_Temp", 4: "HPT_Temp", 5: "Fuel_Flow"}

# Boot the dashboard inline
observe(nerve, launch=True, port=7865)
